In [125]:
import requests
import httpx

EMBEDDING_URL = "http://localhost:8100/embed"

In [126]:
def get_error_list(date: str):
    url = "http://10.122.100.172:9000/analyze/error/list"

    payload = {
        "date": date
    }

    resp = requests.post(url, json=payload)
    return resp.json()

In [127]:
def get_error_file(date: str, file_list: list):
    url = "http://10.122.100.172:9000/analyze/error/file"

    result = []
    for file in file_list :
        
        payload = {
            "date": date,
            "filename" : file
        }

        resp = requests.post(url, json=payload)
        result.append(resp.json())
    return result

In [128]:
date = "2026-04-10"
file_list = get_error_list(date).get("files")
files = get_error_file(date, file_list)

In [129]:
def format_action_plan(action_plan: list[str]) -> str:
    return "\n".join(
        [f"{i+1}. {item}" for i, item in enumerate(action_plan)]
    )

In [130]:
def build_raw_text(file):
    cause_list = file["output_json"]["causeList"]
    error_message = "error_message :\n" + file["input_text"]+"\n\n"
    return error_message+"\n\n".join(
        f"""[CAUSE {i+1}]
title : {c["title"]}
cause : {c["cause"]}
evidence : {c["evidence"]}
actionPlan :
{format_action_plan(c["actionPlan"])}
"""
        for i, c in enumerate(cause_list)
    )

In [131]:
async def embed_texts(texts: list[str]) -> list[list[float]]:
    async with httpx.AsyncClient(timeout=5.0) as client:
        resp = await client.post(
            EMBEDDING_URL,
            json={"texts": texts}
        )
        resp.raise_for_status()
        return resp.json()["dense_vecs"]

In [132]:
def format_action_plan(action_plan: list[str]) -> str:
    return "\n".join(f"{i+1}. {item}" for i, item in enumerate(action_plan))


def build_search_doc(file: dict, date: str, filename: str) -> dict:
    output_json = file.get("output_json") or {}
    cause_list = output_json.get("causeList", [])

    sections = []
    keyword_parts = []
    error_message = file.get("input_text")
    for i, cause in enumerate(cause_list, start=1):
        title = cause.get("title", "")
        cause_text = cause.get("cause", "")
        evidence = cause.get("evidence", "")
        action_plan = cause.get("actionPlan", [])

        sections.append(
            f"""[CAUSE {i}]
title: {title}
cause: {cause_text}
actionPlan:
{format_action_plan(action_plan)}"""
        )

        keyword_parts.append(title)
        keyword_parts.append(evidence)

    vector_text = f"[ERROR MESSAGE]\n{error_message["message"]}\n\n"+"\n\n".join(sections).strip()
    print(vector_text)
    keywords = " ".join(x for x in keyword_parts if x).strip()

    return {
        "date": date,
        "filename": filename,
        "request_id": file.get("request_id"),
        "timestamp": file.get("timestamp"),
        "model_name": file.get("model_name"),
        "error_message": error_message["message"],

        "vector_text": vector_text,
        "keywords": keywords,
    }

In [133]:
INDEX_NAME = "error"

search_docs = []

for i, f in enumerate(files):
    output_json = f.get("output_json") or {}
    cause_list = output_json.get("causeList") or []

    if not cause_list:
        continue

    search_docs.append(
        build_search_doc(file=f, date=date, filename=file_list[i])
    )

vector_text_list = [doc["vector_text"] for doc in search_docs]
embeddings = await embed_texts(vector_text_list)

final_docs = []


for doc, emb in zip(search_docs, embeddings):
    doc["vector"] = emb
    final_docs.append({
        "_op_type": "index",
        "_index": INDEX_NAME,
        "_id": doc["filename"],
        "_source": doc })

[ERROR MESSAGE]
org.springframework.web.util.NestedServletException: Request processing failed; nested exception is java.lang.IllegalStateException: Failed to process checkout request
	at org.springframework.web.servlet.FrameworkServlet.processRequest(FrameworkServlet.java:1014)
	at org.springframework.web.servlet.FrameworkServlet.doPost(FrameworkServlet.java:909)
	at javax.servlet.http.HttpServlet.service(HttpServlet.java:555)
	at org.springframework.web.servlet.FrameworkServlet.service(FrameworkServlet.java:883)
	at javax.servlet.http.HttpServlet.service(HttpServlet.java:623)
	at org.apache.catalina.core.ApplicationFilterChain.internalDoFilter(ApplicationFilterChain.java:209)
	at org.apache.catalina.core.ApplicationFilterChain.doFilter(ApplicationFilterChain.java:153)
Caused by: java.lang.IllegalStateException: Failed to process checkout request
	at com.example.checkout.service.CheckoutService.checkout(CheckoutService.java:148)
	at com.example.checkout.controller.CheckoutController.c

In [134]:
def bulk_insert(docs: list[dict]) -> None:
    from opensearchpy.helpers import bulk
    bulk(client, docs)
    print(f"Inserted {len(docs)} docs")

In [135]:
bulk_insert(final_docs)

Inserted 9 docs


In [81]:
query = ["""
         AttributeError 'NoneType' object has no attribute 'close', anyio/_backends/_asyncio.py
         """]
query_vector = await embed_texts(query)

In [82]:
def search_similar(query: list, query_vector: list, k: int = 3):
    
    body = {
        "size": k,
        "query": {
            "hybrid": {
                "queries": [
                    {
                        "match": {
                            "keyword": query[0]
                        }
                    },
                    {
                        "knn": {
                            "vector": {
                                "vector": query_vector[0],
                                "k": k
                            }
                        }
                    }
                ]
            }
        }
    }

    resp = client.search(index=INDEX_NAME, body=body)
    return resp["hits"]["hits"]

In [83]:
# client.indices.get(index="top_queries-2026.04.09-55111")

In [84]:
OPENSEARCH_HOST = "localhost"
OPENSEARCH_PORT = "9200"
OPENSEARCH_USER = "admin"
OPENSEARCH_PASSWORD = "Dndwls1234!"
OPENSEARCH_USE_SSL = False
OPENSEARCH_VERIFY_CERTS = False
OPENSEARCH_HTTP_COMPRESS = True

EMBEDDING_MODEL = "BAAI/bge-m3"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"


EMBEDDING_URL = "http://localhost:8100/embed"
RERANK_URL = "http://localhost:8100/rerank"
ERROR_LIST_URL = "http://10.122.100.172:9000/analyze/error/list"
ERROR_FILE_URL =" http://10.122.100.172:9000/analyze/error/file"

In [85]:
import os
from functools import lru_cache
from opensearchpy import OpenSearch
# from app.core.config import OPENSEARCH_HOST, OPENSEARCH_PORT, OPENSEARCH_USER, OPENSEARCH_PASSWORD, OPENSEARCH_USE_SSL, OPENSEARCH_VERIFY_CERTS, OPENSEARCH_HTTP_COMPRESS

@lru_cache
def get_opensearch_client() -> OpenSearch:
    kwargs = {
        "hosts": [{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
        "http_compress": OPENSEARCH_HTTP_COMPRESS,
        "use_ssl": OPENSEARCH_USE_SSL,
        "verify_certs": OPENSEARCH_VERIFY_CERTS,
    }

    if OPENSEARCH_PASSWORD:
        kwargs["http_auth"] = (OPENSEARCH_USER, OPENSEARCH_PASSWORD)

    return OpenSearch(**kwargs)

In [86]:
client = get_opensearch_client()

In [87]:
results = search_similar(query, query_vector)

In [88]:
for hit in results:
    print(hit["_score"], hit["_source"]["filename"])

-9549512000.0 20260402_155551_8c80d462
-4422440400.0 20260402_155551_8c80d462
-4422440400.0 20260402_155551_8c80d462


In [89]:
hit

{'_index': 'error',
 '_id': '20260402_155551_8c80d462',
 '_score': -4422440400.0,
 '_source': {'date': '2026-04-02',
  'filename': '20260402_155551_8c80d462',
  'request_id': '8c80d462',
  'timestamp': '2026-04-02T15:56:00.632408+09:00',
  'model_name': 'gpt-oss-20b',
  'error_message': None,
  'vector_text': '[CAUSE 1]\ntitle: GET /favicon.ico 요청에 대한 핸들러 미설정\ncause: 애플리케이션에 /favicon.ico 경로를 처리할 핸들러가 없어서 Spring MVC가 NoHandlerFoundException을 발생시켰다. 이는 일반적으로 정적 리소스가 제공되지 않거나, 리소스 핸들러가 등록되지 않았을 때 발생한다.\nactionPlan:\n1. src/main/resources/static 폴더에 favicon.ico 파일을 추가하고, 웹 서버가 정적 리소스를 제공하도록 설정한다.\n2. Spring Boot의 application.properties에 spring.web.resources.static-locations을 지정해 정적 리소스 경로를 명시한다.\n3. 필요 시, WebMvcConfigurer를 구현해 addResourceHandlers 메서드에서 /favicon.ico 경로를 매핑한다.\n4. 테스트 환경에서 브라우저가 favicon 요청을 보내지 않도록 설정하거나, 404를 무시하도록 로깅 레벨을 조정한다.',
  'keywords': 'GET /favicon.ico 요청에 대한 핸들러 미설정 NoHandlerFoundException No handler found for GET /favicon.ico',
  'vector': [-0.070556640625,
   -0

In [339]:
def make_rerank_text(doc: dict) -> str:
    text = "\n".join([
        str(doc.get("keywords", "") or ""),
        str(doc.get("vector_text", "") or ""),
        str(doc.get("content", "") or "")
    ]).strip()
    return text

In [340]:
import httpx

RERANK_URL = "http://localhost:8100/rerank"


async def rerank_documents(query: str, docs: list[dict], top_n: int = 5) -> list[dict]:
    if not docs:
        return []

    passages = [make_rerank_text(doc) for doc in docs]

    print("docs:", len(docs))
    print("passages:", len(passages))

    async with httpx.AsyncClient(timeout=10.0) as client:
        resp = await client.post(
            RERANK_URL,
            json={
                "query": str(query),
                "passages": passages,
                "max_length": 512,
            },
        )

        print(resp.status_code)
        print(resp.text)

        resp.raise_for_status()
        scores = resp.json()["scores"]

    scored_docs = []
    for doc, score in zip(docs, scores):
        item = dict(doc)
        item["rerank_score"] = float(score)
        scored_docs.append(item)

    scored_docs.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored_docs[:top_n]

In [341]:
reranked = await rerank_documents(query[0], [doc["_source"] for doc in results], top_n=5)

for doc in reranked:
    rag_doc = get_error_file(doc['date'], [doc["filename"]])[0]
    rag_doc['rerank_score'] = doc['rerank_score']
    print(rag_doc)

docs: 3
passages: 3
200
{"scores":[-5.24222469329834,-5.24222469329834,-5.24222469329834]}
{'request_id': '834d8560', 'timestamp': '2026-04-09T13:31:44.838134+09:00', 'client_ip': '10.122.114.176', 'client_port': 63204, 'model_name': 'gpt-oss-20b', 'prompt_tokens': 1350, 'completion_tokens': 1957, 'total_tokens': 3307, 'elapsed_sec': 21.23, 'input_text': {'message': 'java.lang.NullPointerException\n\tat com.example.api.controller.UserController.getUser(UserController.java:87)\n\tat java.base/java.lang.reflect.Method.invoke(Method.java:566)\n\tat org.springframework.web.method.support.InvocableHandlerMethod.doInvoke(InvocableHandlerMethod.java:205)\n\tat org.springframework.web.servlet.DispatcherServlet.doDispatch(DispatcherServlet.java:1072)\nCaused by: org.springframework.dao.DataAccessResourceFailureException: Failed to obtain JDBC Connection\n\tat org.springframework.jdbc.datasource.DataSourceUtils.getConnection(DataSourceUtils.java:82)\n\tat org.springframework.jdbc.core.JdbcTempla

In [298]:
for doc in reranked:
    rag_doc = get_error_file(doc['date'], [doc["filename"]])[0]
    rag_doc['rerank_score'] = doc['rerank_score']
    print(rag_doc)

{'request_id': '4bc70337', 'timestamp': '2026-04-02T20:20:44.423351+09:00', 'client_ip': '10.101.5.160', 'client_port': 36810, 'model_name': 'gpt-oss-20b', 'prompt_tokens': 1954, 'completion_tokens': 623, 'total_tokens': 2577, 'elapsed_sec': 7.415, 'input_text': {'message': 'org.springframework.web.servlet.NoHandlerFoundException: No handler found for GET /favicon.ico\n\tat org.springframework.web.servlet.DispatcherServlet.noHandlerFound(DispatcherServlet.java:1287)\n\tat org.springframework.web.servlet.DispatcherServlet.doDispatch(DispatcherServlet.java:1050)\n\tat org.springframework.web.servlet.DispatcherServlet.doService(DispatcherServlet.java:965)\n\tat org.springframework.web.servlet.FrameworkServlet.processRequest(FrameworkServlet.java:1006)\n\tat org.springframework.web.servlet.FrameworkServlet.doGet(FrameworkServlet.java:898)\n\tat javax.servlet.http.HttpServlet.service(HttpServlet.java:529)\n\tat org.springframework.web.servlet.FrameworkServlet.service(FrameworkServlet.java:8

In [357]:
 resp = client.get(index="error", id="20260409_133123_834d8560")

In [358]:
results

[{'_index': 'error',
  '_id': '20260409_133123_834d8560',
  '_score': -9549512000.0,
  '_source': {'date': '2026-04-09',
   'filename': '20260409_133123_834d8560',
   'request_id': '834d8560',
   'timestamp': '2026-04-09T13:31:44.838134+09:00',
   'model_name': 'gpt-oss-20b',
   'error_message': None,
   'vector_text': '[CAUSE 1]\ntitle: JDBC 연결 실패\ncause: Spring DataAccessResourceFailureException이 발생하여 JDBC 연결을 얻지 못함. 이는 데이터베이스 연결 풀에서 연결을 가져오지 못했음을 의미합니다.\nactionPlan:\n1. 데이터베이스 서비스가 실행 중인지 확인하고, 접속 정보(호스트, 포트, 사용자, 비밀번호)가 올바른지 검증한다.\n2. Spring DataSource 설정(데이터베이스 URL, 드라이버 클래스, 인증 정보)을 재검토한다.\n3. 애플리케이션 로그를 모니터링하여 연결 실패 빈도를 기록하고, 필요 시 알림을 설정한다.\n\n[CAUSE 2]\ntitle: HikariPool 연결 타임아웃\ncause: HikariCP가 지정된 시간(30초) 내에 연결을 확보하지 못해 SQLTransientConnectionException을 발생시킴. 이는 연결 풀의 크기 부족 또는 DB 부하가 원인일 수 있습니다.\nactionPlan:\n1. HikariCP 설정에서 maximumPoolSize, minimumIdle, idleTimeout 등을 조정해 연결 풀 크기를 늘린다.\n2. DB 서버의 연결 수 제한(예: max_connections)을 확인하고 필요 시 증가시킨다.\n3. 애플리케이션에서 연결을 사용한 뒤 반드시 반환하도록

In [300]:
results

[{'_index': 'error',
  '_id': '2026-04-02::20260402_202037_4bc70337::4bc70337',
  '_score': -9549512000.0,
  '_source': {'date': '2026-04-02',
   'filename': '20260402_202037_4bc70337',
   'request_id': '4bc70337',
   'timestamp': '2026-04-02T20:20:44.423351+09:00',
   'model_name': 'gpt-oss-20b',
   'error_message': None,
   'vector_text': '[CAUSE 1]\ntitle: favicon.ico 요청에 대한 핸들러 미설정\ncause: GET /favicon.ico 요청이 들어왔지만, Spring MVC에서 해당 경로를 처리할 핸들러가 없어서 NoHandlerFoundException이 발생했습니다. 이는 보통 정적 리소스 매핑이 없거나, favicon.ico 파일이 존재하지 않을 때 발생합니다.\nactionPlan:\n1. 정적 리소스 매핑을 확인하고, /favicon.ico 경로가 존재하도록 설정합니다.\n2. src/main/resources/static 폴더에 favicon.ico 파일을 추가합니다.\n3. Spring Boot의 application.properties에 spring.web.resources.static-locations을 적절히 설정합니다.\n4. 필요 시 WebMvcConfigurer를 구현해 addResourceHandlers 메서드에서 favicon.ico를 매핑합니다.\n5. 테스트 환경에서 브라우저가 favicon.ico를 정상적으로 요청하고 응답되는지 확인합니다.',
   'keywords': 'favicon.ico 요청에 대한 핸들러 미설정 NoHandlerFoundException No handler found for GET /favicon.ico',
